# 실습 3 · 그리드월드와 Q러닝

**인공지능 일반 · 5차시**

4차시에 손으로 채운 Q표를, 오늘은 **코드가 대신 채웁니다.** 격자는 5×5.

| 순서 | 내용 |
|---|---|
| 1 | 환경 만들기 |
| 2 | Q표 만들기 |
| 3 | ε-그리디 |
| 4 | **Q 업데이트** |
| 5 | 학습 루프 |
| 6 | 결과 보기 |
| 7 | 손잡이 돌려보기 (ε, γ) |

> 실행은 `Shift + Enter`. 위에서부터 순서대로.


## ✏️ 빈칸 5곳

코드에 `여기를_채우세요` 가 5곳 있습니다. 주석의 힌트를 보고 채운 뒤 실행하세요.

| 번호 | 위치 | 무엇을 |
|---|---|---|
| ① | Q표 만들기 | 0으로 가득 찬 (상태 × 행동) 표 |
| ② | ε-그리디 | 탐험할 때 방향 아무거나 |
| ③ | Q 업데이트 | 목표값 `reward + γ · max Q(다음 칸)` |
| ④ | Q 업데이트 | 새 Q값 — 목표값을 그대로 |
| ⑤ | 학습 루프 | 고른다 → 겪는다 → 고친다 → 넘어간다 |

> 빈칸을 채우지 않고 실행하면 `NameError: name '여기를_채우세요' is not defined` 가 납니다.

## 0. 준비

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

# --- 그래프에 한글을 쓰기 위한 준비 (실패해도 실습은 진행됩니다) ---
try:
    import matplotlib.font_manager as fm
    !apt-get -qq install fonts-nanum > /dev/null 2>&1
    fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
    plt.rcParams['font.family'] = 'NanumGothic'
except Exception as e:
    print('한글 폰트 설정 실패 — 그래프의 한글이 □□로 보일 수 있습니다.')

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110

print('준비 완료!')

## 1. 환경 만들기 — 5×5 그리드월드

| 글자 | 뜻 | 보상 |
|---|---|---|
| `S` | 출발점 | – |
| `.` | 빈 칸 | 0 |
| `#` | 벽 (부딪히면 제자리) | 0 |
| `X` | 구덩이 · 판 끝 | **−1** |
| `o` | 작은 보물 · 판 끝 | **+0.2** |
| `G` | 목표 · 판 끝 | **+1** |

`o`는 4걸음, `G`는 8걸음 거리입니다. 가깝고 적은 것 vs 멀고 큰 것 — 3차시 밴딧의 그 딜레마.

AI는 이 지도를 **보지 못합니다.** `step()`이 주는 `(도착 칸, 보상, 끝났나)` 만 보고 배웁니다.


In [ ]:
GRID = [
    "S...o",
    ".##.X",
    ".....",
    "X.##.",
    "....G",
]

ROWS, COLS = len(GRID), len(GRID[0])
N_STATES   = ROWS * COLS          # 25개 칸
MOVES      = {'↑': (-1,  0),      # 위     → 행 −1
              '↓': ( 1,  0),      # 아래   → 행 +1
              '←': ( 0, -1),      # 왼쪽   → 열 −1
              '→': ( 0,  1)}      # 오른쪽 → 열 +1
ACTIONS    = list(MOVES)          # ['↑', '↓', '←', '→']  ← 번호 0,1,2,3 → 화살표
N_ACTIONS  = len(ACTIONS)         # 4

def to_state(row, col):  return row * COLS + col        # (행, 열) → 칸 번호
def to_rowcol(state):    return divmod(state, COLS)     # 칸 번호 → (행, 열)

START = [to_state(row, col) for row in range(ROWS) for col in range(COLS) if GRID[row][col] == 'S'][0]
GOAL  = [to_state(row, col) for row in range(ROWS) for col in range(COLS) if GRID[row][col] == 'G'][0]


def reset():                 # 새 판 시작 → 출발 칸 번호
    return START


# state 칸에서 action 방향으로 한 칸 → (도착 칸 next_state, 보상 reward, 끝났나 done)
def step(state, action):
    row0, col0 = to_rowcol(state)
    d_row, d_col = MOVES[ACTIONS[action]]      # 번호 → 화살표 → 이동량
    row1, col1 = row0 + d_row, col0 + d_col

    # 격자 밖이거나 벽이면 → 제자리
    if not (0 <= row1 < ROWS and 0 <= col1 < COLS) or GRID[row1][col1] == '#':
        row1, col1 = row0, col0

    next_state = to_state(row1, col1)
    cell_char = GRID[row1][col1]

    if cell_char == 'G':
        return next_state,  1.0, True    # 진짜 목표 → 판 끝
    if cell_char == 'o':
        return next_state,  0.2, True    # 작은 보물 → 판 끝
    if cell_char == 'X':
        return next_state, -1.0, True    # 구덩이 → 판 끝
    return next_state, 0.0, False                          # 그 외 → 보상 0, 계속


def argmax_random(values):
    best_value = np.max(values)
    candidates = []
    for idx in range(len(values)):
        if values[idx] == best_value:
            candidates.append(idx)

    return random.choice(candidates)


# 지도를 글자로 출력. robot_state에 칸 번호를 주면 그 자리에 로봇(@)을 그린다
def show_map(robot_state=None):
    for row in range(ROWS):
        line = ''
        for col in range(COLS):
            cell_char = GRID[row][col]
            if robot_state is not None and to_state(row, col) == robot_state:
                line += ' @ '        # @ = 지금 로봇이 서 있는 칸
            elif cell_char == '#':  line += ' ■ '
            elif cell_char == 'X':  line += ' X '
            elif cell_char == 'o':  line += ' o '
            elif cell_char == 'G':  line += ' G '
            else:                   line += ' . '
        print(line)

print(f'칸 {N_STATES}개 × 방향 {N_ACTIONS}개 = Q표 칸 {N_STATES * N_ACTIONS}개')
print(f'출발 s{START}, 목표 s{GOAL} (작은 보물은 4걸음 거리)\n')
show_map(START)

### 손으로 몇 걸음 움직여 보기

`plan`의 화살표를 바꿔가며 실행해 보세요. (`'↑' '↓' '←' '→'`)


In [ ]:
plan = ['↓', '↓', '→', '↓', '↓', '→', '→', '→']   # ← 이 방향들로 차례차례 움직인다

state = reset()
print(f'출발: s{state}')
for arrow in plan:
    action = ACTIONS.index(arrow)      # 화살표 → 번호 (↑0 ↓1 ←2 →3)
    next_state, reward, done = step(state, action)
    mark = ''
    if next_state == state:
        mark = '  ← 벽! 제자리'
    if done:
        mark = '  ← 판 끝!'

    print(f'{ACTIONS[action]}   s{state} → s{next_state}   보상 {reward:g}{mark}')
    state = next_state

    if done:
        break

print()
show_map(state)

## 2. Q표 만들기

**25행 × 4열**, 처음엔 전부 0.


In [ ]:
# 실습 ① : 0으로 가득 찬 (상태 × 행동) 표를 만든다
Q = 여기를_채우세요        # 힌트: np.zeros((행 개수, 열 개수))


def show_q(Q):
    print('      ' + '  '.join(f'{arrow:>5}' for arrow in ACTIONS))
    for state in range(N_STATES):
        values = '  '.join(f'{Q[state, action]:+.2f}' for action in range(N_ACTIONS))
        print(f's{state:<5}{values}')


print('Q표 크기:', Q.shape, '— 칸 25개 × 방향 4개\n')
show_q(Q)

## 3. ε-그리디 — 어느 방향으로 갈까

- 확률 `ε` : 아무 방향이나 (**탐험**)
- 확률 `1−ε` : Q값 1등 방향 (**활용**)

3차시 밴딧과 똑같습니다. 달라진 건 `Q[s]` 로 **상태를 먼저 찾아 들어간다**는 것뿐.


In [ ]:
def choose_action(Q, state, eps):      # state 칸에서 방향 하나 고르기
    if random.random() < eps:
        # ✏️ 실습 ② : 탐험 — 0 ~ 3 중 아무 방향이나 (힌트: random.randrange)
        return 여기를_채우세요
    return argmax_random(Q[state])        # 활용 — Q값 1등


# 확인: 표가 전부 0이면 (=아는 게 없으면) 방향이 골고루 나와야 한다
random.seed(0)
counts = [0, 0, 0, 0]
for _ in range(2000):
    counts[choose_action(Q, START, eps=0.0)] += 1
print('eps=0 인데도 골고루 나옵니다 (전부 동점이라 argmax_random이 무작위로 고름)')
print({ACTIONS[idx]: counts[idx] for idx in range(4)})

## 4. Q 업데이트 — 오늘의 핵심

$$\text{목표} = r + \gamma \cdot \max_{a'} Q(s', a')$$
$$Q(s,a) \leftarrow \text{목표}$$

판이 끝났으면(`done`) 미래가 없으므로 목표는 그냥 `reward`.


In [ ]:
# Q표의 한 칸을 고친다 (표를 제자리에서 직접 수정)
def q_update(Q, state, action, reward, next_state, done, gamma):
    if done:
        target = reward     # 종료 칸 → 미래 없음
    else:
        # ✏️ 실습 ③ : reward + γ × (도착 칸 next_state의 최고 Q값)   힌트: np.max(Q[next_state])
        target = 여기를_채우세요

    # ✏️ 실습 ④ : 목표값을 그대로 새 Q값으로
    Q[state, action] = 여기를_채우세요

### 검증

> 보상 0, 도착 칸의 최고 Q값 1.0, γ=0.9  →  `0 + 0.9 × 1.0 = 0.9`

아래 셀이 에러 없이 지나가면 ③④를 제대로 채운 겁니다.


In [ ]:
testQ = np.zeros((3, 4))
testQ[1, 3] = 1.0          # 1번 칸의 → 방향 Q값이 1.0 이라고 치자

# 0번 칸에서 →(3번 행동)로 가서 1번 칸에 도착, 보상 0, 아직 안 끝남
q_update(testQ, state=0, action=3, reward=0.0, next_state=1, done=False, gamma=0.9)
assert abs(testQ[0, 3] - 0.9) < 1e-9, f'0.9가 나와야 하는데 {testQ[0, 3]} 이 나왔습니다'
print(f'검증 1 통과 :  0 + 0.9 × 1.0 = {testQ[0, 3]}')

# 이번엔 종료 칸(구덩이)에 빠진 경우 → 목표는 그냥 -1
q_update(testQ, state=2, action=1, reward=-1.0, next_state=99, done=True, gamma=0.9)
assert abs(testQ[2, 1] + 1.0) < 1e-9
print(f'검증 2 통과 :  종료 칸이므로 목표 = reward = {testQ[2, 1]}')
print('\n둘 다 통과했습니다. 이제 진짜로 학습시켜 봅시다.')

## 5. 학습 루프 — 1000판 돌리기

```
에피소드를 반복:
    s ← 출발 칸
    판이 끝날 때까지:
        고른다 → 겪는다 → 고친다 → 넘어간다
```

`eps`는 1.0에서 0.05까지 서서히 줄입니다 — 처음엔 헤매고, 나중엔 아는 길로.


In [ ]:
def train(episodes=1000, gamma=0.95,
          eps_start=1.0, eps_end=0.05, max_steps=100, seed=0):

    random.seed(seed); np.random.seed(seed)
    Q = np.zeros((N_STATES, N_ACTIONS))       # 빈 표에서 시작
    returns, lengths = [], []                 # 판마다 점수 / 걸음 수 기록

    for episode in range(episodes):
        # eps를 서서히 줄인다 (앞쪽 70% 구간 동안 직선으로 감소)
        eps = max(eps_end, eps_start - (eps_start - eps_end) * episode / (episodes * 0.7))

        state = reset()
        total_reward = 0.0
        for step_i in range(max_steps):
            # ✏️ 실습 ⑤ : 네 줄을 채워 한 걸음을 완성하세요
            action = 여기를_채우세요                     # 고른다  (choose_action)
            next_state, reward, done = 여기를_채우세요   # 겪는다  (step)
            여기를_채우세요                              # 고친다  (q_update)
            state = 여기를_채우세요                      # 다음 칸으로 넘어간다

            total_reward += reward
            if done:
                break

        returns.append(total_reward)
        lengths.append(step_i + 1)

    return Q, returns, lengths


Q, returns, lengths = train()
print(f'마지막 100판 평균 점수 : {np.mean(returns[-100:]):+.2f}')
print(f'   1.0 = 진짜 목표(G)에 도달  ·  0.2 = 작은 보물(o)에 만족  ·  −1.0 = 구덩이')
print(f'마지막 100판 평균 걸음 : {np.mean(lengths[-100:]):.1f}     (G까지 최단 경로는 8걸음)')

### 학습 곡선

점수가 **0.2 근처에 머물다 1.0으로 올라가는** 구간이 보이면, 작은 보물에 만족하던 AI가 진짜 목표를 발견한 순간입니다.


In [ ]:
def smooth(values, window=50):
    values = np.array(values, dtype=float)
    return np.convolve(values, np.ones(window) / window, mode='valid')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.2))

ax1.plot(smooth(returns), color='#0e8a52', lw=2)
ax1.axhline(1.0, color='#16233c', ls=':', lw=1.2)
ax1.set_title('판당 점수 (50판 이동평균)'); ax1.set_xlabel('에피소드'); ax1.set_ylim(-1.1, 1.2)

ax2.plot(smooth(lengths), color='#0a7ec2', lw=2)
ax2.axhline(8, color='#c8324a', ls=':', lw=1.2)
ax2.set_title('목표까지 걸린 걸음 수 (빨간 점선 = 최단 8걸음)'); ax2.set_xlabel('에피소드')

plt.tight_layout(); plt.show()

print('처음엔 구덩이에 빠지거나 헤매다가, 점점 +1을 8걸음에 받아냅니다.')

## 6. 결과 보기

- **가치 지도** : 칸마다 `max Q(s, ·)`
- **정책 화살표** : 칸마다 Q값 1등 방향 = **길**


In [ ]:
def show_result(Q, title=''):
    value_map = Q.max(axis=1).reshape(ROWS, COLS)          # 칸마다 최고 Q값
    mask = np.zeros((ROWS, COLS), dtype=bool)

    fig, ax = plt.subplots(figsize=(5.2, 5.2))
    heatmap = ax.imshow(value_map, cmap='YlGn',
                        vmin=min(value_map.min(), 0), vmax=max(value_map.max(), 1))

    for row in range(ROWS):
        for col in range(COLS):
            cell_char, state = GRID[row][col], to_state(row, col)
            if cell_char == '#':
                ax.add_patch(plt.Rectangle((col - .5, row - .5), 1, 1, color='#5d6c86'))
                continue
            if cell_char in 'GXo':
                color = {'G': '#0e8a52', 'X': '#c8324a', 'o': '#b06a00'}[cell_char]
                ax.text(col, row, cell_char, ha='center', va='center',
                        fontsize=22, fontweight='bold', color=color)
                continue
            # 가치 숫자
            ax.text(col, row + .34, f'{value_map[row, col]:.2f}', ha='center', va='center',
                    fontsize=8, color='#5d6c86')
            # 최선의 방향을 화살표로
            d_row, d_col = MOVES[ACTIONS[int(np.argmax(Q[state]))]]
            ax.arrow(col - d_col * .18, row - d_row * .18, d_col * .34, d_row * .34,
                     head_width=.16, head_length=.13, fc='#16233c', ec='#16233c', lw=1.4)
            if state == START:
                ax.add_patch(plt.Rectangle((col - .5, row - .5), 1, 1, fill=False,
                                           ec='#0a7ec2', lw=3))

    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title or '가치 지도와 정책 (파란 테두리 = 출발)')
    plt.colorbar(heatmap, ax=ax, shrink=.8, label='max Q(s, ·)')
    plt.tight_layout(); plt.show()


show_result(Q)

**읽는 법**

- G 근처가 진하고 멀수록 옅다 — γ를 계속 곱했으니까
- X 옆 칸의 화살표는 구덩이를 피해 있다
- o 쪽 값도 0보다 크지만 G쪽보다 낮아, 화살표가 o를 지나쳐 간다

이 화살표는 아무도 그려 준 적이 없습니다. **보상 두 개(+1, −1)만 준 결과**입니다.


In [ ]:
# 학습된 Q표로 한 판 (탐험 없이 Q값 1등만 따라감)
def play(Q, verbose=True):
    state = reset(); path = [state]; total_reward = 0.0
    for step_i in range(100):
        action = int(np.argmax(Q[state]))
        state, reward, done = step(state, action)
        path.append(state); total_reward += reward
        if verbose:
            print(f'{step_i+1:2d}걸음  {ACTIONS[action]}  → s{state}')
        if done:
            break
    if verbose:
        print(f'\n{step_i+1}걸음 만에 종료, 점수 {total_reward:+.0f}\n')
        show_map(state)
    return total_reward, step_i + 1

play(Q);

## 7. 손잡이 돌려보기

γ와 ε, 두 손잡이를 실제로 돌려 봅시다.


In [ ]:
# ── ε : 탐험을 아예 안 하면? ──
settings = {
    'ε=0 (탐험 없음)'      : dict(eps_start=0.0, eps_end=0.0),
    'ε=0.1 고정'           : dict(eps_start=0.1, eps_end=0.1),
    'ε 1.0 → 0.05 (감소)'  : dict(eps_start=1.0, eps_end=0.05),
}

plt.figure(figsize=(7.5, 3.2))
for label, kwargs in settings.items():
    scores = []
    for seed in range(5):                      # 씨앗 5개 평균 (운 지우기)
        Q_run, run_returns, _ = train(seed=seed, **kwargs)
        scores.append(smooth(run_returns, 50))
    plt.plot(np.mean(scores, axis=0), lw=2, label=label)

plt.axhline(1.0, color='#16233c', ls=':', lw=1)
plt.title('탐험을 얼마나 할 것인가'); plt.xlabel('에피소드'); plt.ylabel('판당 점수')
plt.legend(); plt.ylim(-1.1, 1.2); plt.tight_layout(); plt.show()

for label, kwargs in settings.items():
    finals = [np.mean(train(seed=seed, **kwargs)[1][-100:]) for seed in range(5)]
    print(f'{label:<22} 마지막 100판 평균 점수 {np.mean(finals):+.2f}')

print()
print('ε=0 은 처음 찾은 작은 보물(+0.2)에 갇혀 나오지 못합니다 — 3차시 그리디의 함정 그대로입니다.')
print('가 본 적 없는 길의 Q값은 0이라, 0.2짜리 길이 늘 이겨 버리니까요.')

In [ ]:
# ── γ : 얼마나 멀리 볼 것인가 ──
print('γ        출발 칸의 값    학습 후 경로     G 도달률')
print('-' * 46)
for gamma in [0.5, 0.9, 0.95, 0.99, 1.0]:
    path_lengths, wins, start_values = [], [], []
    for seed in range(5):                       # 씨앗 5개 평균
        Q_gamma, _, _ = train(gamma=gamma, seed=seed)
        total_reward, steps = play(Q_gamma, verbose=False)
        path_lengths.append(steps)
        wins.append(total_reward > 0.5)             # 0.2(작은 보물) 말고 진짜 G만
        start_values.append(Q_gamma[START].max())
    print(f'{gamma:<9}{np.mean(start_values):8.3f}{np.mean(path_lengths):13.1f}걸음{np.mean(wins)*100:10.0f}%')

print()
print('γ=0.5 : 출발 칸의 값이 거의 0 — 8걸음 거리의 +1이 여기까지 닿지 못합니다.')
print('        그래서 4걸음짜리 작은 보물(+0.2)에 만족합니다. 근시안적인 AI입니다.')
print('γ=1.0 : 돌아가는 길과 지름길의 값이 똑같이 1.0 — 최단 경로를 고집할 이유가 없어져')
print('        100걸음을 헤매다 시간이 끝납니다. γ<1이 곧 "빨리 가라"는 압력이었던 겁니다.')
print()
print('γ는 "얼마나 멀리 보는가"입니다. 너무 가까이 보면 눈앞의 것에 만족하고,')
print('무한히 멀리 보면 서두를 이유가 사라집니다.')

show_result(train(gamma=0.5)[0], title='γ = 0.5 — 가까운 작은 보물로 화살표가 몰린다')

---

## 도전 과제

1. **지도 직접 그리기** — `GRID` 글자를 바꾸면 새 환경입니다. 목표가 도달 불가능하면 어떻게 되나요?

2. **작은 보물을 0.8로** — `step()`에서 `0.2` → `0.8`. ε을 아무리 조절해도 G로 안 가는 지점은?
   (힌트: G의 가치는 출발 기준 0.95⁷ ≈ 0.70)

3. **걸음마다 −0.01** — `return next_state, 0.0, False` → `return next_state, -0.01, False`.
   γ=1.0 이어도 최단 경로를 찾나요? 왜일까요?

4. **미끄러지는 빙판** — `step()` 첫 줄에 `if random.random() < 0.2: action = random.randrange(4)`.
   AI는 여전히 G를 찾아내나요? 학습 곡선이 왜 들쭉날쭉해질까요?

5. **Q표 눈으로 보기**
   ```python
   import pandas as pd
   pd.DataFrame(Q, columns=ACTIONS).round(3)
   ```

6. **표의 한계** — 우리 표는 100칸이었습니다. 바둑처럼 상태가 10¹⁷⁰개면 표를 만들 수 없습니다.
   그때 표 대신 신경망에게 Q값을 계산시키는 것이 **DQN** 입니다. 나머지는 오늘 것 그대로.
